# LeetCode 533 — Lonely Pixel II bằng Python

Link đề: https://leetcode.com/problems/lonely-pixel-ii/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Matrix

> **Ghi chú Premium:** đây là bài **Premium** (🔒). Đề trong notebook được viết theo hiểu biết về bài; nếu bạn có tài khoản Premium, hãy **đối chiếu lại đề gốc** (nhất là câu chữ của ràng buộc và hai ví dụ).

Skeleton LeetCode:

```python
class Solution:
    def findBlackPixel(self, picture: List[List[str]], target: int) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đếm các ô `'B'` thỏa **hai luật** — hàng và cột chứa đúng `target` ô đen, và **mọi hàng có ô đen ở cột đó phải giống hệt hàng đang xét**.
2. Nắm các công cụ Python cần cho bài ma trận: `picture[i][j]`, `row.count("B")`, `zip(*picture)` để lấy cột, `Counter`/`defaultdict(list)`, so sánh cả hàng bằng `==`, biến hàng thành khóa băm (`tuple`, `"".join`), và **bitmask** (`1 << j`, `bit_count`).
3. Đi từ **vét cạn từng ô** (`O(m²n²)`) → **đếm hàng + gom hàng theo cột** (`O(mn²)`) → **gom các hàng giống hệt bằng `Counter`** (`O(mn)`) → **bitmask hai chiều** (so sánh cả tập hàng bằng một phép `==` trên số nguyên).
4. Thấy được mấu chốt: **mọi ô `B` trong cùng một cột có chung kết luận**, nên ta xét **theo cột**, mỗi cột hợp lệ cộng đúng `target`.
5. Tránh các lỗi kinh điển: quên kiểm tra hàng "mẫu" có đủ `target` ô đen, quên điều kiện hàng giống nhau, cộng `1` thay vì `target`, `g[j][0]` trên cột rỗng, dùng `list` làm khóa dict, và dựng ma trận test bằng `[["W"] * n] * m` (các hàng chung một đối tượng).

## 2. Đề bài & ví dụ

Cho ma trận `picture` kích thước `m x n` gồm các pixel đen `'B'` và trắng `'W'`, và một số nguyên `target`. Hãy trả về **số lượng pixel đen "cô đơn"**.

Một pixel đen tại vị trí `(r, c)` là **cô đơn** nếu:

1. Hàng `r` và cột `c` **đều chứa đúng `target`** pixel đen.
2. Với **mọi hàng có pixel đen ở cột `c`**, hàng đó phải **giống hệt** hàng `r` (cả `n` ký tự).

**Ví dụ 1** (`target = 3`):

```
picture = [["W","B","W","B","B","W"],
           ["W","B","W","B","B","W"],
           ["W","B","W","B","B","W"],
           ["W","W","B","W","B","W"]]
Output: 6
```

Giải thích: các ô `B` ở **cột 1 và cột 3** (mỗi cột 3 ô, ở hàng 0, 1, 2) là cô đơn. Lấy `(0, 1)`: hàng 0 có 3 `B`, cột 1 có 3 `B`; các hàng có `B` ở cột 1 là hàng 0, 1, 2 — cả ba giống hệt hàng 0.

**Ví dụ 2** (`target = 1`):

```
picture = [["W","W","B"],
           ["W","W","B"],
           ["W","W","B"]]
Output: 0
```

Cột 2 có tới 3 ô `B` (khác `target = 1`) nên không ô nào cô đơn.

**Ràng buộc:** `1 <= m, n <= 200`; `picture[i][j]` là `'W'` hoặc `'B'`; `1 <= target <= min(m, n)`.

In [1]:
from collections import Counter, defaultdict
from typing import List
import itertools
import random
import time


def P(*rows):
    """Tạo ma trận (list các list ký tự) từ các chuỗi, ví dụ P("WB", "BW")."""
    return [list(r) for r in rows]


ex1 = P("WBWBBW", "WBWBBW", "WBWBBW", "WWBWBW")
ex2 = P("WWB", "WWB", "WWB")

for row in ex1:
    print(" ".join(row))
print("m, n =", len(ex1), len(ex1[0]), "| target = 3 | đáp án mong đợi: 6")
print()
for row in ex2:
    print(" ".join(row))
print("m, n =", len(ex2), len(ex2[0]), "| target = 1 | đáp án mong đợi: 0")

W B W B B W
W B W B B W
W B W B B W
W W B W B W
m, n = 4 6 | target = 3 | đáp án mong đợi: 6

W W B
W W B
W W B
m, n = 3 3 | target = 1 | đáp án mong đợi: 0


## 3. Kiến thức nền

### 3.1. Ma trận là list của các list

`picture[i]` là **hàng thứ `i`** (một list ký tự), `picture[i][j]` là ô ở hàng `i`, cột `j`. Số hàng `m = len(picture)`, số cột `n = len(picture[0])`.

In [2]:
pic = P("WBW", "BBW")
print(pic)
print("m =", len(pic), "n =", len(pic[0]))
print("hàng 1:", pic[1], "| ô (1, 0):", pic[1][0], "| ô (0, 2):", pic[0][2])

[['W', 'B', 'W'], ['B', 'B', 'W']]
m = 2 n = 3
hàng 1: ['B', 'B', 'W'] | ô (1, 0): B | ô (0, 2): W


### 3.2. Đếm ô đen trong một hàng

Hàng là list ký tự nên `row.count("B")` đếm số `'B'`. Muốn đếm theo **cột** thì phải duyệt cả cột (hoặc tích lũy khi duyệt từng hàng).

In [3]:
pic = P("WBW", "BBW")
print([row.count("B") for row in pic])              # đếm theo hàng -> [1, 2]

col_cnt = [0] * len(pic[0])
for row in pic:
    for j, ch in enumerate(row):
        if ch == "B":
            col_cnt[j] += 1
print(col_cnt)                                      # đếm theo cột -> [1, 2, 0]

[1, 2]
[1, 2, 0]


### 3.3. Lấy cột bằng `zip(*picture)`

`zip(*picture)` "xoay" ma trận: phần tử đầu của mọi hàng gộp thành cột 0, ... Mỗi cột là một `tuple`. Đây là cách ngắn gọn để đếm theo cột.

In [4]:
pic = P("WBW", "BBW")
cols = list(zip(*pic))
print(cols)                                         # [('W','B'), ('B','B'), ('W','W')]
print([c.count("B") for c in cols])                 # [1, 2, 0]

[('W', 'B'), ('B', 'B'), ('W', 'W')]
[1, 2, 0]


### 3.4. So sánh cả hàng và biến hàng thành khóa băm

Hai list so sánh bằng `==` theo từng phần tử: `["W","B"] == ["W","B"]` là `True`. Nhưng **list không băm được** nên không làm khóa dict/Counter được. Hai cách chuẩn: `tuple(row)` hoặc `"".join(row)` (với ký tự đơn như `'B'`/`'W'`).

In [5]:
a, b = list("WBB"), list("WBB")
print(a == b, a is b)                               # True False (bằng giá trị, khác đối tượng)
try:
    {a: 1}
except TypeError as e:
    print("list làm khóa ->", e)

print(Counter(tuple(r) for r in P("WB", "WB", "BW")))
print(Counter("".join(r) for r in P("WB", "WB", "BW")))

True False
list làm khóa -> unhashable type: 'list'
Counter({('W', 'B'): 2, ('B', 'W'): 1})
Counter({'WB': 2, 'BW': 1})


### 3.5. `Counter`, `defaultdict(list)` và `all(...)`

- `Counter(iterable)`: đếm số lần xuất hiện của mỗi giá trị.
- `defaultdict(list)`: dict tự tạo list rỗng khi truy cập khóa mới — tiện để "gom" các chỉ số theo khóa (ví dụ: cột → danh sách hàng có `B`).
- `all(cond for x in xs)`: `True` nếu **mọi** phần tử thỏa; với dãy rỗng cũng là `True`.

In [6]:
g = defaultdict(list)
for i, row in enumerate(P("WB", "BB", "WB")):
    for j, ch in enumerate(row):
        if ch == "B":
            g[j].append(i)
print(dict(g))                                      # {1: [0, 1, 2], 0: [1]}
print(0 in g, 5 in g, len(g))                       # khóa chưa từng có thì không có trong g (khi chỉ dùng `in`)

print(all(x % 2 == 0 for x in [2, 4, 6]), all(x % 2 == 0 for x in [2, 3]), all(x > 0 for x in []))

{1: [0, 1, 2], 0: [1]}
True False 2
True False True


### 3.6. Bitmask: một tập hợp nhỏ lưu trong một số nguyên

Tập chỉ số `{0, 2}` có thể lưu thành số nguyên có bit 0 và bit 2 bật: `(1 << 0) | (1 << 2) = 5`.

- `1 << j`: số có đúng bit `j` bật. `mask |= 1 << j`: thêm `j` vào tập.
- `mask.bit_count()`: số phần tử của tập (số bit 1). (Python 3.10+; cũ hơn dùng `bin(mask).count("1")`.)
- `mask & -mask`: giữ lại **bit thấp nhất**; `.bit_length() - 1` cho chỉ số của bit đó.
- Hai tập bằng nhau **⇔** hai mask bằng nhau — một phép `==` thay cho duyệt từng phần tử.

In [7]:
mask = 0
for j in (0, 2, 5):
    mask |= 1 << j
print(mask, bin(mask), "| số phần tử:", mask.bit_count(), bin(mask).count("1"))
low = mask & -mask
print("bit thấp nhất:", low, "-> chỉ số", low.bit_length() - 1)
print((1 << 0 | 1 << 2) == 5)

37 0b100101 | số phần tử: 3 3
bit thấp nhất: 1 -> chỉ số 0
True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy lấy **Ví dụ 1** và tự hỏi: ô `B` ở `(0, 1)` có cô đơn không?

```
      c0 c1 c2 c3 c4 c5
 r0:   W  B  W  B  B  W      3 ô B
 r1:   W  B  W  B  B  W      3 ô B
 r2:   W  B  W  B  B  W      3 ô B
 r3:   W  W  B  W  B  W      2 ô B
```

1. **Hàng 0** có 3 `B` = `target` ✓.
2. **Cột 1** có `B` ở hàng 0, 1, 2 → 3 ô = `target` ✓.
3. Các hàng có `B` ở cột 1 là hàng 0, 1, 2. So với hàng 0: hàng 1 = `WBWBBW` ✓, hàng 2 ✓ → cả ba giống hệt ✓.

Vậy `(0, 1)` cô đơn. Hoàn toàn tương tự `(1, 1)`, `(2, 1)`. Còn cột 4? Có `B` ở **cả 4 hàng** (0, 1, 2, 3) → 4 ô ≠ 3 → không ai trong cột 4 cô đơn. Cột 2: chỉ hàng 3 có `B`, mà hàng 3 chỉ có 2 `B` ≠ 3 → không cô đơn.

**Quan sát then chốt:** ta chưa hề nhìn từng ô riêng lẻ mà đã kết luận cho **cả cột**: `(0, 1), (1, 1), (2, 1)` có cùng số phận. Cột 1 và cột 3 hợp lệ → mỗi cột góp `target = 3` ô → `3 + 3 = 6`. ✓

## 5. Cách 1 — Brute Force / baseline: kiểm tra từng ô theo đúng định nghĩa

**Ý tưởng:** duyệt mọi ô. Với mỗi ô `B` ở `(r, c)`:

1. Đếm `B` của hàng `r` — phải bằng `target`.
2. Đếm `B` của cột `c` — phải bằng `target`.
3. Với **mọi hàng `i`** có `picture[i][c] == "B"`, so `picture[i] == picture[r]`.

Nếu cả ba đều đạt thì tăng đáp án thêm 1. Đây là bản dịch **nguyên văn** của đề, nên nó cũng chính là **oracle** (đáp án chuẩn) để đối chiếu các cách sau.

In [8]:
def lonely_brute(picture, target):
    m, n = len(picture), len(picture[0])
    ans = 0
    for r in range(m):
        for c in range(n):
            if picture[r][c] != "B":
                continue
            row_black = sum(1 for j in range(n) if picture[r][j] == "B")
            col_black = sum(1 for i in range(m) if picture[i][c] == "B")
            if row_black != target or col_black != target:
                continue
            ok = True
            for i in range(m):
                if picture[i][c] == "B" and picture[i] != picture[r]:
                    ok = False
                    break
            if ok:
                ans += 1
    return ans


print(lonely_brute(ex1, 3), lonely_brute(ex2, 1))
assert lonely_brute(ex1, 3) == 6 and lonely_brute(ex2, 1) == 0

6 0


**Độ phức tạp:** có `m·n` ô; mỗi ô `B` tốn `O(n)` đếm hàng + `O(m)` đếm cột + tới `O(m·n)` so sánh các hàng (mỗi lần so hàng tốn `O(n)`, có thể so `m` hàng). Tổng `O(m·n·(m·n))` = **`O(m²n²)`**. Với `m = n = 200` là cỡ `1.6 × 10⁹` — quá chậm. Space `O(1)`.

**Nút thắt:** ta lặp đi lặp lại **cùng một việc** cho mọi ô trong một cột (cùng đếm cột, cùng so các hàng đó) và đếm lại hàng cho mọi ô trong cùng một hàng. Cần (a) tính trước số `B` mỗi hàng/cột, (b) kết luận **một lần cho cả cột**.

## 6. Ý tưởng thuật toán / bước đột phá

**Bổ đề 1 — mọi ô `B` trong một cột có chung số phận.** Gọi `R_c` là tập các hàng có `B` ở cột `c`. Ô `(r, c)` với `r ∈ R_c` cô đơn khi và chỉ khi:

- `cột c` có đúng `target` ô: `|R_c| = target`;
- **mọi** hàng trong `R_c` giống hệt hàng `r` — suy ra **mọi hàng trong `R_c` giống nhau** — nên hàng `r` nào cũng như nhau, và số `B` của hàng `r` là một giá trị chung `k`; cần `k = target`.

Nếu tập hàng `R_c` không phải toàn hàng giống nhau thì **không** `r` nào thỏa; nếu toàn giống nhau thì **mọi** `r ∈ R_c` cùng thỏa hoặc cùng trượt. Vậy ta chỉ cần kiểm tra **một hàng đại diện** rồi cộng `target` (đúng bằng số ô `B` của cột) nếu cột hợp lệ.

**Bổ đề 2 — gom hàng giống nhau.** Nếu cột `c` hợp lệ với hàng đại diện `r`, thì **mọi hàng giống hệt `r` đều có `B` ở cột `c`**, nên chúng đều nằm trong `R_c`. Mà `R_c` chỉ gồm các hàng giống `r` ⇒ `R_c` = *đúng nhóm các hàng giống `r`*, và nhóm đó có **đúng `target` hàng**. Do đó:

> Một hàng mẫu `s` (một "khuôn" chuỗi) cho ra ô cô đơn khi và chỉ khi **`s` xuất hiện đúng `target` lần** và **`s` có đúng `target` ô `B`**. Khi đó, mỗi cột `j` mà `s[j] == "B"` **và** cột `j` có đúng `target` ô `B` đều đóng góp `target` vào đáp án.

Từ hai bổ đề, ta có các cách làm nhanh: **gom hàng theo cột** (Cách 2), **đếm các hàng giống nhau bằng `Counter`** (Cách 3), và **biểu diễn tập hàng bằng bitmask** (Cách 4).

## 7. Các cách tiếp theo

Tổng cộng 4 cách: Cách 1 (vét cạn, ở mục 5) và ba cách nhanh dưới đây. Ba cách đều **khác nhau thật về ý tưởng**: Cách 2 đi **theo cột** và so sánh từng hàng; Cách 3 đi **theo hàng**, nhóm các hàng giống nhau bằng bảng băm; Cách 4 biến mỗi tập hàng thành **một số nguyên** để so sánh bằng một phép `==`.

### Cách 2 — Đếm hàng + gom hàng theo cột (`defaultdict(list)`)

**Giải thích từng đoạn:**

1. `rows[i]` = số `B` của hàng `i`; `g[j]` = danh sách các hàng có `B` ở cột `j` (chính là `R_j`).
2. Duyệt các cột `j` **có** `B` (các khóa của `g`). Lấy hàng đầu tiên `i1 = g[j][0]` làm đại diện.
3. Nếu `rows[i1] != target`: bỏ cột. Nếu `len(g[j]) != target`: bỏ cột (cột không đúng `target` ô).
4. Nếu mọi hàng trong `g[j]` bằng `picture[i1]`: cột hợp lệ, `ans += target`.

In [9]:
def lonely_adj(picture, target):
    m, n = len(picture), len(picture[0])
    rows = [0] * m                       # số B của từng hàng
    g = defaultdict(list)                # g[j] = các hàng có B ở cột j
    for i in range(m):
        for j in range(n):
            if picture[i][j] == "B":
                rows[i] += 1
                g[j].append(i)

    ans = 0
    for j in g:                          # chỉ các cột có ít nhất một B
        i1 = g[j][0]                     # hàng đại diện
        if rows[i1] != target or len(g[j]) != target:
            continue
        if all(picture[i2] == picture[i1] for i2 in g[j]):
            ans += target                # cả cột hợp lệ: target ô cô đơn
    return ans


print(lonely_adj(ex1, 3), lonely_adj(ex2, 1))
assert lonely_adj(ex1, 3) == 6 and lonely_adj(ex2, 1) == 0

6 0


**Code nộp LeetCode (Cách 2):**

```python
class Solution:
    def findBlackPixel(self, picture: List[List[str]], target: int) -> int:
        m, n = len(picture), len(picture[0])
        rows = [0] * m
        g = defaultdict(list)
        for i in range(m):
            for j in range(n):
                if picture[i][j] == "B":
                    rows[i] += 1
                    g[j].append(i)
        ans = 0
        for j in g:
            i1 = g[j][0]
            if rows[i1] != target or len(g[j]) != target:
                continue
            if all(picture[i2] == picture[i1] for i2 in g[j]):
                ans += target
        return ans
```

**Độ phức tạp:** xây `rows` và `g`: `O(mn)`. Mỗi cột hợp lệ tới bước so sánh: có đúng `target ≤ m` hàng cần so, mỗi lần so `O(n)` ⇒ `O(m·n)` mỗi cột; `n` cột ⇒ **`O(m·n²)`** (chặn trên; thực tế nhỏ hơn vì chỉ cột qua hai bộ lọc đầu mới so). Space `O(mn)` cho `g` (tối đa mỗi ô `B` là một phần tử).

### Cách 3 — Gom các hàng giống hệt bằng `Counter` (theo hàng)

**Giải thích từng đoạn:**

1. Đếm số `B` mỗi cột: `col_cnt[j]`.
2. Biến mỗi hàng thành chuỗi `"".join(row)` và `Counter` chúng: `same[s]` = số hàng đúng bằng `s`.
3. Với mỗi khuôn `s` có `same[s] == target` **và** `s.count("B") == target` (Bổ đề 2): mọi cột `j` có `s[j] == "B"` mà `col_cnt[j] == target` đóng góp `target`.

Không cần so sánh hàng với hàng nữa: bảng băm đã "so" giúp ta.

In [10]:
def lonely_group(picture, target):
    m, n = len(picture), len(picture[0])
    col_cnt = [0] * n
    for row in picture:
        for j, ch in enumerate(row):
            if ch == "B":
                col_cnt[j] += 1

    same = Counter("".join(row) for row in picture)   # khuôn hàng -> số hàng giống hệt
    ans = 0
    for s, times in same.items():
        if times != target or s.count("B") != target:
            continue
        for j, ch in enumerate(s):
            if ch == "B" and col_cnt[j] == target:
                ans += target
    return ans


print(lonely_group(ex1, 3), lonely_group(ex2, 1))
assert lonely_group(ex1, 3) == 6 and lonely_group(ex2, 1) == 0

6 0


**Code nộp LeetCode (Cách 3):**

```python
class Solution:
    def findBlackPixel(self, picture: List[List[str]], target: int) -> int:
        n = len(picture[0])
        col_cnt = [0] * n
        for row in picture:
            for j, ch in enumerate(row):
                if ch == "B":
                    col_cnt[j] += 1
        same = Counter("".join(row) for row in picture)
        ans = 0
        for s, times in same.items():
            if times != target or s.count("B") != target:
                continue
            for j, ch in enumerate(s):
                if ch == "B" and col_cnt[j] == target:
                    ans += target
        return ans
```

**Độ phức tạp:** đếm cột `O(mn)`; tạo `m` chuỗi dài `n` và băm chúng `O(mn)`; duyệt các khuôn: có tối đa `m` khuôn, mỗi khuôn quét `n` ký tự ⇒ `O(mn)`. Tổng **`O(mn)`**, Space `O(mn)` (các chuỗi khuôn).

### Cách 4 — Bitmask: so sánh cả tập hàng bằng một phép `==`

**Giải thích từng đoạn:**

1. `row_mask[i]`: số nguyên có bit `j` bật nếu `picture[i][j] == "B"` (tập cột đen của hàng `i`). `col_mask[j]`: bit `i` bật nếu `picture[i][j] == "B"` (tập hàng đen của cột `j`, chính là `R_j`).
2. `group[row_mask]`: gom các hàng có cùng mask thành **một mask hàng**: bit `i` bật nếu hàng `i` có khuôn đó.
3. Với mỗi cột `j` không rỗng: lấy hàng đại diện `i1` (bit thấp nhất của `col_mask[j]`). Cột hợp lệ ⇔ `col_mask[j] == group[row_mask[i1]]` (tập hàng đen của cột **đúng bằng** nhóm hàng giống `i1`, theo Bổ đề 2), kèm `|R_j| = target` và hàng `i1` có đúng `target` ô `B`.

In [11]:
def lonely_bitmask(picture, target):
    m, n = len(picture), len(picture[0])
    row_mask = [0] * m
    col_mask = [0] * n
    for i in range(m):
        for j in range(n):
            if picture[i][j] == "B":
                row_mask[i] |= 1 << j
                col_mask[j] |= 1 << i

    group = {}                                        # khuôn hàng -> mask các hàng có khuôn đó
    for i in range(m):
        group[row_mask[i]] = group.get(row_mask[i], 0) | (1 << i)

    ans = 0
    for j in range(n):
        cm = col_mask[j]
        if cm == 0:
            continue
        i1 = (cm & -cm).bit_length() - 1              # hàng đại diện = hàng đen thấp nhất
        rm = row_mask[i1]
        if cm.bit_count() == target and rm.bit_count() == target and group[rm] == cm:
            ans += target
    return ans


print(lonely_bitmask(ex1, 3), lonely_bitmask(ex2, 1))
assert lonely_bitmask(ex1, 3) == 6 and lonely_bitmask(ex2, 1) == 0

6 0


**Code nộp LeetCode (Cách 4):**

```python
class Solution:
    def findBlackPixel(self, picture: List[List[str]], target: int) -> int:
        m, n = len(picture), len(picture[0])
        row_mask = [0] * m
        col_mask = [0] * n
        for i in range(m):
            for j in range(n):
                if picture[i][j] == "B":
                    row_mask[i] |= 1 << j
                    col_mask[j] |= 1 << i
        group = {}
        for i in range(m):
            group[row_mask[i]] = group.get(row_mask[i], 0) | (1 << i)
        ans = 0
        for j in range(n):
            cm = col_mask[j]
            if cm == 0:
                continue
            i1 = (cm & -cm).bit_length() - 1
            rm = row_mask[i1]
            if cm.bit_count() == target and rm.bit_count() == target and group[rm] == cm:
                ans += target
        return ans
```

**Độ phức tạp:** dựng mask `O(mn)` (mỗi `|=` trên số nguyên `≤ 200` bit coi như `O(n/64)` hay hằng); gom nhóm `O(m)` phép dict (băm số nguyên `n` bit, `O(n/64)`); duyệt cột `O(n)`. Tổng **`O(mn)`**, Space `O(m + n)` số nguyên lớn (tương đương `O(mn/64)` từ máy), nhỏ nhất trong các cách.

### Kiểm tra nhanh bốn cách trên hai ví dụ và một "bẫy"

Bẫy: ba hàng giống nhau `BW` và `target = 2` — các hàng đều có 1 `B` và cột 0 có tận 3 ô `B` (thừa một hàng) nên **không** hợp lệ; hơn nữa mỗi hàng chỉ có 1 `B` ≠ 2. Cả bốn cách phải cùng cho `0`.

In [12]:
trap = P("BW", "BW", "BW")         # target = 2: cột 0 có 3 B (thừa một hàng)
IMPLS = {"brute": lonely_brute, "adj": lonely_adj, "group": lonely_group, "mask": lonely_bitmask}

for name, f in IMPLS.items():
    print(f"{name:<6}", f(ex1, 3), f(ex2, 1), f(trap, 2))
    assert (f(ex1, 3), f(ex2, 1), f(trap, 2)) == (6, 0, 0)

brute  6 0 0
adj    6 0 0
group  6 0 0
mask   6 0 0


## 8. TRACE từng bước

**Trace Cách 2 trên Ví dụ 1 (`target = 3`):**

| Bước | Dữ liệu | Kết luận |
|---|---|---|
| Đếm hàng | `rows = [3, 3, 3, 2]` | hàng 3 chỉ có 2 `B` |
| Gom cột | `g[1] = [0,1,2]`, `g[2] = [3]`, `g[3] = [0,1,2]`, `g[4] = [0,1,2,3]` | cột 0 và 5 không có `B` nên không xuất hiện |
| Cột 1 | `i1 = 0`, `rows[0] = 3 = target`, `len(g[1]) = 3 = target`, các hàng 0,1,2 giống nhau | hợp lệ, `ans = 3` |
| Cột 2 | `i1 = 3`, `rows[3] = 2 ≠ 3` | bỏ |
| Cột 3 | giống cột 1 (hàng 0,1,2) | hợp lệ, `ans = 6` |
| Cột 4 | `i1 = 0`, `rows[0] = 3`, nhưng `len(g[4]) = 4 ≠ 3` | bỏ |

Hàm `trace_adj` dưới đây in đúng các bước đó; sau đó `trace_group` in cách nhìn **theo hàng** (Cách 3).

In [13]:
def trace_adj(picture, target):
    m, n = len(picture), len(picture[0])
    rows = [0] * m
    g = defaultdict(list)
    for i in range(m):
        for j in range(n):
            if picture[i][j] == "B":
                rows[i] += 1
                g[j].append(i)
    print("rows =", rows)
    print("g    =", {j: g[j] for j in sorted(g)})
    ans = 0
    for j in sorted(g):
        i1 = g[j][0]
        print(f"- cột {j}: hàng đại diện i1={i1}, rows[i1]={rows[i1]}, |g[j]|={len(g[j])}", end=" -> ")
        if rows[i1] != target:
            print("bỏ (hàng đại diện không có đúng target B)")
            continue
        if len(g[j]) != target:
            print("bỏ (cột không có đúng target B)")
            continue
        same = all(picture[i2] == picture[i1] for i2 in g[j])
        if same:
            ans += target
            print(f"HỢP LỆ, ans += {target} -> {ans}")
        else:
            print("bỏ (có hàng khác hàng đại diện)")
    print("kết quả:", ans, "\n")
    return ans


assert trace_adj(ex1, 3) == 6
assert trace_adj(ex2, 1) == 0
assert trace_adj(trap, 2) == 0

rows = [3, 3, 3, 2]
g    = {1: [0, 1, 2], 2: [3], 3: [0, 1, 2], 4: [0, 1, 2, 3]}
- cột 1: hàng đại diện i1=0, rows[i1]=3, |g[j]|=3 -> HỢP LỆ, ans += 3 -> 3
- cột 2: hàng đại diện i1=3, rows[i1]=2, |g[j]|=1 -> bỏ (hàng đại diện không có đúng target B)
- cột 3: hàng đại diện i1=0, rows[i1]=3, |g[j]|=3 -> HỢP LỆ, ans += 3 -> 6
- cột 4: hàng đại diện i1=0, rows[i1]=3, |g[j]|=4 -> bỏ (cột không có đúng target B)
kết quả: 6 

rows = [1, 1, 1]
g    = {2: [0, 1, 2]}
- cột 2: hàng đại diện i1=0, rows[i1]=1, |g[j]|=3 -> bỏ (cột không có đúng target B)
kết quả: 0 

rows = [1, 1, 1]
g    = {0: [0, 1, 2]}
- cột 0: hàng đại diện i1=0, rows[i1]=1, |g[j]|=3 -> bỏ (hàng đại diện không có đúng target B)
kết quả: 0 



In [14]:
def trace_group(picture, target):
    n = len(picture[0])
    col_cnt = [sum(1 for row in picture if row[j] == "B") for j in range(n)]
    print("col_cnt =", col_cnt)
    same = Counter("".join(row) for row in picture)
    print("same    =", dict(same))
    ans = 0
    for s, times in same.items():
        print(f"- khuôn {s}: xuất hiện {times} lần, có {s.count('B')} B", end=" -> ")
        if times != target or s.count("B") != target:
            print("bỏ")
            continue
        good = [j for j, ch in enumerate(s) if ch == "B" and col_cnt[j] == target]
        ans += target * len(good)
        print(f"các cột hợp lệ {good}, cộng {target}*{len(good)} -> ans = {ans}")
    print("kết quả:", ans, "\n")
    return ans


assert trace_group(ex1, 3) == 6
assert trace_group(ex2, 1) == 0
assert trace_group(P("BBWW", "BBWW", "WWBB", "WWBB"), 2) == 8

col_cnt = [0, 3, 1, 3, 4, 0]
same    = {'WBWBBW': 3, 'WWBWBW': 1}
- khuôn WBWBBW: xuất hiện 3 lần, có 3 B -> các cột hợp lệ [1, 3], cộng 3*2 -> ans = 6
- khuôn WWBWBW: xuất hiện 1 lần, có 2 B -> bỏ
kết quả: 6 

col_cnt = [0, 0, 3]
same    = {'WWB': 3}
- khuôn WWB: xuất hiện 3 lần, có 1 B -> bỏ
kết quả: 0 

col_cnt = [2, 2, 2, 2]
same    = {'BBWW': 2, 'WWBB': 2}
- khuôn BBWW: xuất hiện 2 lần, có 2 B -> các cột hợp lệ [0, 1], cộng 2*2 -> ans = 4
- khuôn WWBB: xuất hiện 2 lần, có 2 B -> các cột hợp lệ [2, 3], cộng 2*2 -> ans = 8
kết quả: 8 



**Trace Cách 4 (bitmask) cho `ex1`** — in ra các mask dưới dạng nhị phân để thấy tập hợp được so sánh thế nào (bit 0 ở **bên phải**):

In [15]:
def trace_mask(picture, target):
    m, n = len(picture), len(picture[0])
    row_mask, col_mask = [0] * m, [0] * n
    for i in range(m):
        for j in range(n):
            if picture[i][j] == "B":
                row_mask[i] |= 1 << j
                col_mask[j] |= 1 << i
    group = {}
    for i in range(m):
        group[row_mask[i]] = group.get(row_mask[i], 0) | (1 << i)
    print("row_mask:", [format(x, f"0{n}b") for x in row_mask])
    print("col_mask:", [format(x, f"0{m}b") for x in col_mask])
    print("group   :", {format(k, f"0{n}b"): format(v, f"0{m}b") for k, v in group.items()})
    ans = 0
    for j in range(n):
        cm = col_mask[j]
        if cm == 0:
            print(f"- cột {j}: rỗng"); continue
        i1 = (cm & -cm).bit_length() - 1
        rm = row_mask[i1]
        ok = cm.bit_count() == target and rm.bit_count() == target and group[rm] == cm
        print(f"- cột {j}: col_mask={cm:0{m}b}, đại diện i1={i1}, group[row_mask[i1]]={group[rm]:0{m}b}, hợp lệ={ok}")
        ans += target if ok else 0
    print("kết quả:", ans)
    return ans


assert trace_mask(ex1, 3) == 6

row_mask: ['011010', '011010', '011010', '010100']
col_mask: ['0000', '0111', '1000', '0111', '1111', '0000']
group   : {'011010': '0111', '010100': '1000'}
- cột 0: rỗng
- cột 1: col_mask=0111, đại diện i1=0, group[row_mask[i1]]=0111, hợp lệ=True
- cột 2: col_mask=1000, đại diện i1=3, group[row_mask[i1]]=1000, hợp lệ=False
- cột 3: col_mask=0111, đại diện i1=0, group[row_mask[i1]]=0111, hợp lệ=True
- cột 4: col_mask=1111, đại diện i1=0, group[row_mask[i1]]=0111, hợp lệ=False
- cột 5: rỗng
kết quả: 6


## 9. Code nộp LeetCode

Ta nộp **Cách 3 (Counter)** vì ngắn, nhanh `O(mn)` và dễ giải thích; nếu LeetCode chưa tự import, thêm `from collections import Counter` và `from typing import List` ở đầu file.

In [16]:
class Solution:
    def findBlackPixel(self, picture: List[List[str]], target: int) -> int:
        n = len(picture[0])
        col_cnt = [0] * n
        for row in picture:
            for j, ch in enumerate(row):
                if ch == "B":
                    col_cnt[j] += 1

        same = Counter("".join(row) for row in picture)
        ans = 0
        for s, times in same.items():
            if times != target or s.count("B") != target:
                continue
            for j, ch in enumerate(s):
                if ch == "B" and col_cnt[j] == target:
                    ans += target
        return ans


sol = Solution()
print(sol.findBlackPixel(ex1, 3), sol.findBlackPixel(ex2, 1))
assert sol.findBlackPixel(ex1, 3) == 6 and sol.findBlackPixel(ex2, 1) == 0
IMPLS["Solution"] = lambda p, t: Solution().findBlackPixel(p, t)

6 0


## 10. Vì sao đúng

Gọi `R_j` là tập hàng có `B` ở cột `j`, và `rows[i]`, `col_cnt[j]` như trên.

- **Điều kiện của đề cho ô `(r, c)`:** `rows[r] = target`, `col_cnt[c] = |R_c| = target`, và mọi `i ∈ R_c` có `picture[i] = picture[r]`.
- **Gộp theo cột:** nếu ô `(r, c)` thỏa thì mọi `i ∈ R_c` đều giống `r`, do đó đều giống nhau và `rows[i] = target`; vậy mọi ô `(i, c)` với `i ∈ R_c` cũng thỏa. Ngược lại nếu một ô nào đó của cột không thỏa thì cả cột không thỏa (vì điều kiện "mọi hàng trong `R_c` giống `r`" ép các hàng bằng nhau). Do đó **số ô cô đơn của cột `c` hoặc bằng `0` hoặc bằng `|R_c| = target`**.
- **Cách 3 đúng:** nếu cột `j` hợp lệ với đại diện `s = picture[i1]` thì `s[j] = "B"`, `col_cnt[j] = target`, `rows[i1] = s.count("B") = target`, và mọi hàng bằng `s` đều có `B` ở cột `j` nên nằm trong `R_j`; vì `R_j` toàn hàng bằng `s` nên **số hàng bằng `s` đúng `= |R_j| = target`**. Ngược lại, nếu `s` xuất hiện đúng `target` lần, `s.count("B") = target`, `s[j] = "B"`, `col_cnt[j] = target` thì `R_j` chứa nhóm `target` hàng bằng `s` mà `|R_j| = target` nên `R_j` **chính là** nhóm đó ⇒ cột hợp lệ.
- **Không đếm trùng cột:** một cột `j` hợp lệ chỉ ứng với **một** khuôn `s` (hàng đại diện duy nhất của `R_j`); hai khuôn khác nhau cùng có `B` ở `j` sẽ làm `|R_j| ≥ 2·target > target`.
- **Cách 4** là Cách 2/3 viết bằng tập: `R_j == group(i1)` là đúng mệnh đề "tập hàng đen của cột bằng nhóm hàng giống `i1`".

Ngoài lập luận, mục 12 đối chiếu bốn cách với vét cạn trên toàn bộ ma trận nhỏ và hàng nghìn ma trận ngẫu nhiên.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Vét cạn từng ô | `O(m²n²)` | `O(1)` | oracle, dùng cho test |
| 2. Gom hàng theo cột | `O(m·n²)` | `O(mn)` | dễ hiểu, theo sát đề |
| 3. `Counter` các hàng giống nhau | `O(mn)` | `O(mn)` | **nộp bài**, tuyến tính theo kích thước ma trận |
| 4. Bitmask | `O(mn)` | `O(m + n)` số nguyên lớn | so sánh tập bằng `==` |

Với `m, n ≤ 200`, ma trận có tối đa `40 000` ô: cả Cách 2, 3, 4 chạy tức thì; chỉ Cách 1 là nguy hiểm trong trường hợp xấu. Mục 12 có thử thời gian trên ma trận `200 x 200`.

## 12. Bộ test

Chiến lược:

1. **Ví dụ đề** (2 ví dụ) cho cả bốn cách + `Solution`.
2. **Edge case:** ma trận `1x1`, toàn `W`, toàn `B`, `target = min(m, n)`, một hàng/một cột, cột chỉ có `B` ở hàng không đủ `target`.
3. **Phản ví dụ** cho các lời giải sai hay gặp (chỉ đếm hàng/cột; quên kiểm tra hàng đại diện) — dùng ở mục 13.
4. **Vét cạn toàn bộ** mọi ma trận `3x3`, `3x4`, `4x3`, `2x5` với mọi `target` hợp lệ: so bốn cách với oracle.
5. **Ngẫu nhiên có seed:** ma trận ngẫu nhiên và ma trận có cấu trúc (nhóm hàng giống nhau) đúng ràng buộc; đảm bảo có đủ nhiều đáp án khác 0.
6. **Thời gian** trên `200 x 200`.

In [17]:
def check_all(picture, target, expected=None):
    ref = lonely_brute(picture, target)
    if expected is not None:
        assert ref == expected, (picture, target, ref, expected)
    for name, f in IMPLS.items():
        got = f(picture, target)
        assert got == ref, (name, picture, target, got, ref)
    return ref


# 1) Ví dụ đề
check_all(ex1, 3, 6)
check_all(ex2, 1, 0)

# 2) Edge case
check_all(P("B"), 1, 1)                           # 1x1 đen
check_all(P("W"), 1, 0)                           # 1x1 trắng
check_all(P("BB", "BB"), 2, 4)                    # toàn B, target = 2: cả 2 cột hợp lệ => 2 cột * 2 = 4
check_all(P("BB", "BB"), 1, 0)                    # toàn B nhưng target = 1: mỗi hàng/cột có 2 B
check_all(P("WWW", "WWW"), 1, 0)                  # toàn W
check_all(P("BWB"), 1, 0)                         # một hàng: hàng có 2 B, cột có 1 B => hàng sai
check_all(P("BWW"), 1, 1)                         # một hàng, một B
check_all(P("B", "W", "B"), 1, 0)                 # một cột: cột có 2 B
check_all(P("B", "W", "B"), 2, 0)                 # một cột 2 B nhưng mỗi hàng chỉ có 1 B
check_all(P("BW", "WB"), 1, 2)                    # hai ô cô đơn chéo
check_all(P("BWB", "BWB", "WBW"), 2, 4)           # nhóm 2 hàng giống nhau, 2 cột
print("Edge case: OK")

Edge case: OK


In [18]:
# 4) Vét cạn: mọi ma trận m x n nhỏ với mọi target hợp lệ
def all_pictures(m, n):
    for bits in itertools.product("WB", repeat=m * n):
        yield [list(bits[i * n:(i + 1) * n]) for i in range(m)]


total = nonzero = 0
for m, n in [(3, 3), (3, 4), (4, 3), (2, 5), (1, 6), (6, 1)]:
    for pic in all_pictures(m, n):
        for t in range(1, min(m, n) + 1):
            r = check_all(pic, t)
            total += 1
            nonzero += r > 0
print(f"Vét cạn: {total} cặp (ma trận, target), {nonzero} cặp có đáp án > 0 — bốn cách trùng oracle")

Vét cạn: 28288 cặp (ma trận, target), 2361 cặp có đáp án > 0 — bốn cách trùng oracle


In [19]:
# 5) Ngẫu nhiên có seed
def gen_random(rng):
    m, n = rng.randint(1, 7), rng.randint(1, 7)
    p = rng.choice([0.2, 0.5, 0.8])
    pic = [["B" if rng.random() < p else "W" for _ in range(n)] for _ in range(m)]
    return pic, rng.randint(1, min(m, n))


def gen_structured(rng):
    """Dựng các nhóm `t` hàng giống hệt nhau, mỗi nhóm có `t` cột đen, rồi thêm ít nhiễu."""
    m, n = rng.randint(2, 9), rng.randint(2, 9)
    t = rng.randint(1, min(m, n))
    pic = [["W"] * n for _ in range(m)]
    pool = list(range(m))
    while len(pool) >= t and rng.random() < 0.9:
        rs = rng.sample(pool, t)
        cs = rng.sample(range(n), t)
        for r in rs:
            pool.remove(r)
            for c in cs:
                pic[r][c] = "B"
    for _ in range(rng.randint(0, 2)):
        r, c = rng.randrange(m), rng.randrange(n)
        pic[r][c] = "W" if pic[r][c] == "B" else "B"
    return pic, t


rng = random.Random(2024)
cnt, pos = 0, 0
for gen in (gen_random, gen_structured):
    for _ in range(3000):
        pic, t = gen(rng)
        assert 1 <= len(pic) <= 200 and 1 <= len(pic[0]) <= 200 and 1 <= t <= min(len(pic), len(pic[0]))
        r = check_all(pic, t)
        cnt += 1
        pos += r > 0
print(f"Ngẫu nhiên: {cnt} test, {pos} test có đáp án > 0 — OK")
assert pos > 500, "cần đủ test có đáp án khác 0"

Ngẫu nhiên: 6000 test, 1899 test có đáp án > 0 — OK


In [20]:
# 6) Thời gian: g nhóm, mỗi nhóm t hàng giống nhau và t cột riêng => đáp án = g * t * t
def build_blocks(g, t):
    size = g * t
    pic = [["W"] * size for _ in range(size)]
    for k in range(g):
        for r in range(k * t, (k + 1) * t):
            for c in range(k * t, (k + 1) * t):
                pic[r][c] = "B"
    return pic


big = build_blocks(10, 20)           # 200 x 200, target = 20
for name in ("adj", "group", "mask", "Solution"):
    t0 = time.perf_counter()
    ans = IMPLS[name](big, 20)
    dt = (time.perf_counter() - t0) * 1000
    assert ans == 10 * 20 * 20
    print(f"{name:<9} m=n=200 -> {ans}   ({dt:.1f} ms)")

small = build_blocks(6, 5)           # 30 x 30: vét cạn còn chạy được để so sánh
assert lonely_brute(small, 5) == 6 * 5 * 5 and all(f(small, 5) == 150 for f in IMPLS.values())

# ma trận toàn B kích thước k x k, target = k: vét cạn phải "đi hết" mọi ô nên thời gian tăng rất nhanh
for k in (25, 50, 100):
    full = [["B"] * k for _ in range(k)]
    t0 = time.perf_counter(); a1 = lonely_brute(full, k); d1 = (time.perf_counter() - t0) * 1000
    t0 = time.perf_counter(); a2 = lonely_group(full, k); d2 = (time.perf_counter() - t0) * 1000
    assert a1 == a2 == k * k
    print(f"toàn B {k:>3}x{k:<3}: brute {d1:8.1f} ms | group {d2:6.2f} ms")

# ma trận toàn B 200 x 200: cột nào cũng có 200 B
allb = [["B"] * 200 for _ in range(200)]
for name in ("adj", "group", "mask"):
    assert IMPLS[name](allb, 200) == 200 * 200 == IMPLS["Solution"](allb, 200)
    assert IMPLS[name](allb, 100) == 0
print("200x200 toàn B: OK")

adj       m=n=200 -> 4000   (3.0 ms)
group     m=n=200 -> 4000   (1.7 ms)
mask      m=n=200 -> 4000   (2.3 ms)
Solution  m=n=200 -> 4000   (2.3 ms)
toàn B  25x25 : brute      4.3 ms | group   0.05 ms
toàn B  50x50 : brute     32.9 ms | group   0.24 ms


toàn B 100x100: brute    259.7 ms | group   0.54 ms
200x200 toàn B: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: chỉ kiểm tra "hàng và cột đều có `target` ô đen" — quên điều kiện hàng giống nhau

```python
def wrong1(picture, target):
    ...
    # chỉ đếm B có rows[r] == target và cols[c] == target
```

Vì sao sai: luật 2 của đề bị bỏ. Hai hàng có cùng số `B` vẫn có thể **khác vị trí** `B`.

In [21]:
def wrong_counts_only(picture, target):
    m, n = len(picture), len(picture[0])
    rows = [row.count("B") for row in picture]
    cols = [col.count("B") for col in zip(*picture)]
    return sum(1 for r in range(m) for c in range(n)
               if picture[r][c] == "B" and rows[r] == target and cols[c] == target)


pic = P("BBW", "BWB", "WWW")          # target = 2: cột 0 có 2 B ở hàng 0 và 1, nhưng hai hàng KHÁC nhau
print("sai :", wrong_counts_only(pic, 2))
print("đúng:", lonely_group(pic, 2), "(oracle:", lonely_brute(pic, 2), ")")
assert wrong_counts_only(pic, 2) == 2 and lonely_group(pic, 2) == 0

sai : 2
đúng: 0 (oracle: 0 )


Cách sửa: thêm điều kiện **mọi hàng trong `R_c` bằng hàng đại diện** (Cách 2) hoặc yêu cầu khuôn hàng xuất hiện đúng `target` lần (Cách 3).

### Lỗi 2: quên kiểm tra hàng đại diện có đúng `target` ô đen

Chỉ kiểm tra cột có `target` ô và các hàng của cột giống nhau — nhưng hàng đó có thể có **nhiều hơn** `target` ô đen.

In [22]:
def wrong_no_row_check(picture, target):
    g = defaultdict(list)
    for i, row in enumerate(picture):
        for j, ch in enumerate(row):
            if ch == "B":
                g[j].append(i)
    ans = 0
    for j in g:
        i1 = g[j][0]
        if len(g[j]) == target and all(picture[i2] == picture[i1] for i2 in g[j]):
            ans += target                         # thiếu: rows[i1] == target
    return ans


pic = P("BB", "WW")                               # target = 1: hàng 0 có 2 B => không ô nào cô đơn
print("sai :", wrong_no_row_check(pic, 1))
print("đúng:", lonely_adj(pic, 1), "(oracle:", lonely_brute(pic, 1), ")")
assert wrong_no_row_check(pic, 1) == 2 and lonely_adj(pic, 1) == 0

sai : 2
đúng: 0 (oracle: 0 )


### Lỗi 3: cộng `1` cho mỗi cột hợp lệ thay vì `target`

Mỗi cột hợp lệ có `target` ô cô đơn (một ô ở mỗi hàng của nhóm). Nếu chỉ `ans += 1` thì đáp án nhỏ hơn thật.

In [23]:
def wrong_plus_one(picture, target):
    g = defaultdict(list)
    rows = [row.count("B") for row in picture]
    for i, row in enumerate(picture):
        for j, ch in enumerate(row):
            if ch == "B":
                g[j].append(i)
    ans = 0
    for j in g:
        i1 = g[j][0]
        if rows[i1] == target and len(g[j]) == target and all(picture[i2] == picture[i1] for i2 in g[j]):
            ans += 1                              # sai: phải cộng target
    return ans


print("sai :", wrong_plus_one(ex1, 3), "(đáp án thật 6)")
print("đúng:", lonely_adj(ex1, 3))
assert wrong_plus_one(ex1, 3) == 2 and lonely_adj(ex1, 3) == 6

sai : 2 (đáp án thật 6)
đúng: 6


### Lỗi 4: lấy `g[j][0]` trên cột **không có** ô đen (`IndexError`)

Nếu duyệt `for j in range(n)` rồi truy cập `g[j][0]` mà `g` là `defaultdict(list)`, cột rỗng sẽ tự tạo `[]` và `[][0]` ném `IndexError`. Duyệt `for j in g` (chỉ cột có `B`) hoặc kiểm tra rỗng trước.

In [24]:
def wrong_index_error(picture, target):
    n = len(picture[0])
    g = defaultdict(list)
    for i, row in enumerate(picture):
        for j, ch in enumerate(row):
            if ch == "B":
                g[j].append(i)
    ans = 0
    for j in range(n):
        i1 = g[j][0]                              # cột rỗng -> IndexError
        ans += 0
    return ans


try:
    wrong_index_error(P("WB"), 1)
except IndexError as e:
    print("IndexError:", e)
print("đúng:", lonely_adj(P("WB"), 1))
assert lonely_adj(P("WB"), 1) == 1

IndexError: list index out of range
đúng: 1


### Lỗi 5: dùng `list` làm khóa `Counter`/`dict`, và dựng test bằng `[["W"] * n] * m`

- `Counter(picture)` với `picture` là list của list: **`TypeError: unhashable type: 'list'`** — đổi hàng thành `tuple(row)` hoặc `"".join(row)`.
- `[["W"] * n] * m` tạo `m` tham chiếu tới **cùng một hàng**: sửa một ô là sửa cả cột. Dùng `[["W"] * n for _ in range(m)]`.

In [25]:
try:
    Counter(P("WB", "BW"))
except TypeError as e:
    print("TypeError:", e)

bad = [["W"] * 3] * 2                             # hai hàng là CÙNG một list
bad[0][1] = "B"
print("dựng sai :", bad)                          # cả hai hàng đều bị sửa

good = [["W"] * 3 for _ in range(2)]
good[0][1] = "B"
print("dựng đúng:", good)
assert bad[1][1] == "B" and good[1][1] == "W"

TypeError: unhashable type: 'list'
dựng sai : [['W', 'B', 'W'], ['W', 'B', 'W']]
dựng đúng: [['W', 'B', 'W'], ['W', 'W', 'W']]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Pattern "điều kiện lặp cho mọi phần tử của nhóm":** khi đề đòi một điều kiện đúng cho *mọi* ô trong cùng hàng/cột, hãy tự hỏi: cả nhóm có chung kết luận không? Nếu có, **xét một lần cho cả nhóm** (ở đây: một cột) thay vì từng ô.
- **"So sánh nguyên cả hàng"** thường được thay bằng **băm** (`tuple`, `"".join`, mask nguyên) rồi đếm bằng `Counter`: hai hàng giống nhau ⇔ cùng khóa băm. So sánh từng cặp là `O(m²n)`, băm chỉ `O(mn)`.
- **Đếm hàng/cột trước, quyết định sau:** nhiều bài ma trận (Lonely Pixel I, Set Matrix Zeroes, Valid Sudoku) dùng hai mảng đếm `rows[]`, `cols[]`.
- **Bitmask cho tập nhỏ** (`≤ 200` phần tử vẫn ổn với số nguyên Python): so sánh/gom tập bằng `==`/`|`/`&`.
- **Nhìn hình học:** các ô cô đơn tạo thành **khối**: một nhóm đúng `target` hàng giống hệt nhau × các cột đen *hợp lệ* của khuôn hàng đó. Số ô của khối = `target × (số cột hợp lệ)`. Trong `ex1`: nhóm 3 hàng × 2 cột hợp lệ (cột 1 và cột 3) = `3 × 2 = 6`; cột 4 cũng là cột đen của khuôn nhưng bị loại vì có 4 ô `B` (hàng 3 chen vào).
- **Dò nhanh bằng "tập hợp":** `Counter` cho biết khuôn nào xuất hiện đúng `target` lần — chỉ những khuôn đó đáng xét.

## 15. Biến thể của bài

1. **LeetCode 531 — Lonely Pixel I:** `B` cô đơn nếu **không còn `B` nào khác** trong cùng hàng và cột (tương đương `target = 1` ở luật 1 và bỏ luật 2). Đếm `rows`, `cols` rồi đếm ô.
2. **Trả về danh sách tọa độ** các ô cô đơn của bài 533 (không chỉ số lượng).
3. **`target` thay đổi theo hàng** hoặc cho phép "ít nhất `target`" thay vì "đúng `target`".
4. **Ma trận rất lớn / luồng dữ liệu:** không giữ được cả ma trận — chỉ giữ `rows`, `cols` và băm hàng.
5. **Ma trận ba chiều hoặc nhiều màu:** khuôn hàng là chuỗi nhiều ký tự; cách `Counter` giữ nguyên.

Cài nhanh biến thể 1 (Lonely Pixel I) và đối chiếu với vét cạn:

In [26]:
def lonely_pixel_one(picture):
    """LeetCode 531: B cô đơn = hàng và cột của nó đều chỉ có đúng 1 B."""
    rows = [row.count("B") for row in picture]
    cols = [col.count("B") for col in zip(*picture)]
    return sum(1 for r, row in enumerate(picture) for c, ch in enumerate(row)
               if ch == "B" and rows[r] == 1 and cols[c] == 1)


def lonely_pixel_one_brute(picture):
    m, n = len(picture), len(picture[0])
    out = 0
    for r in range(m):
        for c in range(n):
            if picture[r][c] == "B":
                others = [(i, j) for i in range(m) for j in range(n)
                          if picture[i][j] == "B" and (i, j) != (r, c) and (i == r or j == c)]
                out += not others
    return out


rng = random.Random(5)
for _ in range(500):
    pic, _t = gen_random(rng)
    assert lonely_pixel_one(pic) == lonely_pixel_one_brute(pic)
print(lonely_pixel_one(P("WWB", "WBW", "BWW")), lonely_pixel_one(P("BBW", "WWB")))
# Với target=1 của bài 533, các cột hợp lệ cũng chính là Lonely Pixel I (hai luật trùng nhau)
rng = random.Random(6)
for _ in range(500):
    pic, _t = gen_random(rng)
    assert lonely_group(pic, 1) == lonely_pixel_one(pic)
print("Lonely Pixel I = bài 533 với target = 1: OK")

3 1
Lonely Pixel I = bài 533 với target = 1: OK


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn từng ô | dịch nguyên văn đề, kiểm tra từng ô `B` | `O(m²n²)` | `O(1)` | làm oracle để test, ma trận rất nhỏ |
| 2. Gom hàng theo cột | `rows[]` + `g[j]`; kiểm một cột bằng so sánh hàng | `O(mn²)` | `O(mn)` | muốn code sát lời đề, dễ chứng minh |
| 3. `Counter` các hàng | khuôn hàng giống nhau xuất hiện đúng `target` lần | `O(mn)` | `O(mn)` | **cách nộp**: ngắn, nhanh, dễ nhớ |
| 4. Bitmask | tập hàng của cột `==` nhóm hàng giống | `O(mn)` | `O(m+n)` số lớn | khi muốn so tập hợp nhanh, bài có kích thước vừa phải |

Trực giác chọn: nếu thấy "hàng giống hệt nhau", **băm hàng** (Cách 3/4); nếu thấy "mọi phần tử của nhóm", **xét theo nhóm** (Cách 2).

## 17. 5 bài tập vận dụng (dễ → khó)

Mọi cell TODO **không gọi** hàm khi chạy notebook; đáp án có assert ở mục 19.

### Bài 1 (đoán output)

Đoán trước, rồi chạy để kiểm tra.

```python
pic = P("BW", "BW", "WB")
print(lonely_group(pic, 1))
print(lonely_group(pic, 2))
```

In [27]:
pic_b1 = P("BW", "BW", "WB")
print(lonely_group(pic_b1, 1))
print(lonely_group(pic_b1, 2))

1
0


### Bài 2 (tính tay)

`picture` dưới đây, `target = 2`. Hãy tự tính: các hàng có bao nhiêu `B`? cột nào hợp lệ? đáp án bằng bao nhiêu? Ghi vào biến `my_answer_2` (cell dưới để `None`), đối chiếu ở mục 19.

```
picture = ["BWB",
           "BWB",
           "WBW"]
```

In [28]:
pic_b2 = P("BWB", "BWB", "WBW")
my_answer_2 = None          # TODO: điền đáp án tính tay, ví dụ my_answer_2 = 0

### Bài 3 (điền TODO)

Hoàn thành hàm tính `rows` (số `B` mỗi hàng) và `cols` (số `B` mỗi cột). Bạn đừng gọi hàm khi chưa viết xong.

In [29]:
def count_rows_cols(picture):
    # TODO: trả về (rows, cols); rows[i] = số B của hàng i, cols[j] = số B của cột j
    # Gợi ý: dùng row.count("B") và zip(*picture)
    rows = None
    cols = None
    return rows, cols

### Bài 4 (tự viết lại)

Tự viết `my_find_black_pixel(picture, target)` theo **Cách 2** (gom hàng theo cột bằng `defaultdict(list)`) mà không nhìn code cũ. Cell dưới chỉ định nghĩa hàm rỗng.

In [30]:
def my_find_black_pixel(picture, target):
    # TODO: viết lại Cách 2 (rows[] + g[j] + so sánh hàng đại diện)
    pass

### Bài 5 (biến thể / mini challenge)

Viết `lonely_positions(picture, target)` trả về **danh sách tọa độ `(r, c)` đã sắp xếp** của mọi ô cô đơn (thay vì chỉ đếm). Gợi ý: cột hợp lệ cho ra các ô `(i, j)` với `i ∈ R_j`. Đáp án phải thỏa `len(list) == lonely_group(...)`.

In [31]:
def lonely_positions(picture, target):
    # TODO: trả về list các (r, c), đã sắp xếp
    pass

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):**
- Bài 1: mỗi hàng có bao nhiêu `B`? Hai hàng đầu giống nhau. Thử `target = 1`: cột 0 có mấy `B`?
- Bài 2: đếm `B` mỗi hàng và mỗi cột, rồi xét cột nào có hai hàng giống nhau.
- Bài 3: `row.count("B")` đếm một hàng; `zip(*picture)` cho từng cột.
- Bài 4: bốn bước: đếm `rows`; gom `g[j]`; với mỗi cột lấy `i1 = g[j][0]`; kiểm tra điều kiện.
- Bài 5: sửa Cách 2: khi cột hợp lệ, thêm `(i, j)` cho mọi `i` trong `g[j]`.

**Mức 2 (vừa):**
- Bài 1: cột 0 có `B` ở hàng 0 và 1 (hai hàng giống nhau, mỗi hàng 1 `B`); cột 1 chỉ có `B` ở hàng 2.
- Bài 2: hàng 0, 1 giống `BWB` (2 `B`), hàng 2 là `WBW` (1 `B`). Cột 0 và cột 2 đều có `B` ở hàng 0, 1.
- Bài 3: `[row.count("B") for row in picture]` và `[col.count("B") for col in zip(*picture)]`.
- Bài 4: điều kiện cột hợp lệ gồm `rows[i1] == target`, `len(g[j]) == target`, `all(picture[i2] == picture[i1] ...)`.
- Bài 5: sắp xếp bằng `sorted(...)` ở cuối.

**Mức 3 (gần đáp án):**
- Bài 1: với `target = 1`: cột 0 có 2 `B` ≠ 1 → loại; cột 1 có 1 `B` ở hàng 2, hàng 2 có 1 `B` → hợp lệ → `1`. Với `target = 2`: cột 0 có 2 `B` ở hàng 0, 1 nhưng mỗi hàng chỉ có 1 `B` ≠ 2 → loại → `0`.
- Bài 2: cột 0 và 2 đều hợp lệ (cả hai: 2 `B`, hai hàng giống nhau và mỗi hàng 2 `B`): `2 + 2 = 4`.
- Bài 3: trả về `(rows, cols)` với hai list comprehension ở trên.
- Bài 4: copy cấu trúc `lonely_adj` nhưng tự gõ lại từng dòng.
- Bài 5: trong vòng lặp cột hợp lệ: `out.extend((i, j) for i in g[j])`.

## 19. Đáp án tham khảo

Phần này đặt cuối bài giảng; mọi đáp án có assert.

In [32]:
# Bài 1
assert lonely_group(pic_b1, 1) == 1
assert lonely_group(pic_b1, 2) == 0
print("Bài 1: 1 và 0")

# Bài 2
ans_2 = 4
assert lonely_brute(pic_b2, 2) == ans_2 == lonely_group(pic_b2, 2)
print("Bài 2:", ans_2, "(cột 0 và cột 2 hợp lệ, mỗi cột 2 ô)")

# Bài 3
def count_rows_cols_sol(picture):
    rows = [row.count("B") for row in picture]
    cols = [col.count("B") for col in zip(*picture)]
    return rows, cols


assert count_rows_cols_sol(pic_b2) == ([2, 2, 1], [2, 1, 2])
assert count_rows_cols_sol(ex1) == ([3, 3, 3, 2], [0, 3, 1, 3, 4, 0])
print("Bài 3:", count_rows_cols_sol(ex1))

Bài 1: 1 và 0
Bài 2: 4 (cột 0 và cột 2 hợp lệ, mỗi cột 2 ô)
Bài 3: ([3, 3, 3, 2], [0, 3, 1, 3, 4, 0])


In [33]:
# Bài 4
def my_find_black_pixel_sol(picture, target):
    m, n = len(picture), len(picture[0])
    rows = [0] * m
    g = defaultdict(list)
    for i in range(m):
        for j in range(n):
            if picture[i][j] == "B":
                rows[i] += 1
                g[j].append(i)
    ans = 0
    for j in g:
        i1 = g[j][0]
        if rows[i1] == target and len(g[j]) == target and all(picture[i2] == picture[i1] for i2 in g[j]):
            ans += target
    return ans


rng = random.Random(11)
for _ in range(300):
    pic, t = gen_structured(rng)
    assert my_find_black_pixel_sol(pic, t) == lonely_brute(pic, t)
print("Bài 4: OK")


# Bài 5
def lonely_positions_sol(picture, target):
    m, n = len(picture), len(picture[0])
    rows = [row.count("B") for row in picture]
    g = defaultdict(list)
    for i in range(m):
        for j in range(n):
            if picture[i][j] == "B":
                g[j].append(i)
    out = []
    for j in g:
        i1 = g[j][0]
        if rows[i1] == target and len(g[j]) == target and all(picture[i2] == picture[i1] for i2 in g[j]):
            out.extend((i, j) for i in g[j])
    return sorted(out)


assert lonely_positions_sol(ex1, 3) == [(0, 1), (0, 3), (1, 1), (1, 3), (2, 1), (2, 3)]
assert lonely_positions_sol(ex2, 1) == []
rng = random.Random(12)
for _ in range(300):
    pic, t = gen_structured(rng)
    pos_list = lonely_positions_sol(pic, t)
    assert len(pos_list) == lonely_brute(pic, t)
    assert all(pic[r][c] == "B" for r, c in pos_list)
print("Bài 5:", lonely_positions_sol(ex1, 3))

Bài 4: OK
Bài 5: [(0, 1), (0, 3), (1, 1), (1, 3), (2, 1), (2, 3)]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template (Cách 3, `Counter` các hàng):**

```python
class Solution:
    def findBlackPixel(self, picture, target):
        col_cnt = [0] * len(picture[0])
        for row in picture:
            for j, ch in enumerate(row):
                if ch == "B":
                    col_cnt[j] += 1
        same = Counter("".join(row) for row in picture)
        ans = 0
        for s, times in same.items():
            if times == target and s.count("B") == target:       # khuôn hợp lệ
                for j, ch in enumerate(s):
                    if ch == "B" and col_cnt[j] == target:       # cột cũng đúng target
                        ans += target
        return ans
```

**Công thức nhớ:** *khuôn hàng xuất hiện đúng `target` lần, có đúng `target` ô `B`; mỗi cột `B` của nó mà cột đó đúng `target` ô ⇒ cộng `target`.*

**Checklist trước khi submit:**

- [ ] Đã đọc đủ **hai luật**: đúng `target` ở hàng **và** cột, **và** các hàng có `B` ở cột đó giống hệt.
- [ ] Cộng **`target`** cho mỗi cột hợp lệ (không phải `1`, không phải số cột).
- [ ] Kiểm tra cả `rows[đại diện] == target` lẫn `|cột| == target`.
- [ ] Không truy cập `g[j][0]` trên cột rỗng; không dùng `list` làm khóa dict.
- [ ] Test: `1x1`, toàn `W`, toàn `B`, `target = min(m, n)`, một hàng/một cột, bẫy "hàng khác nhau nhưng cùng số `B`".
- [ ] Đã so với vét cạn trên input nhỏ và chạy `200 x 200` (nhanh).
- [ ] Bài Premium: đã đối chiếu đề gốc nếu có tài khoản.